# Evolution strategies (CMA-ES) — runnable notebook
One focused concept, **5 runnable & visualizable examples** — each cell computes, plots, and asserts. Run-all safe.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, math, itertools
np.random.seed(0)
def sphere(x, target=None):
    x=np.asarray(x, dtype=float)
    if target is None: target=np.zeros_like(x)
    return float(np.sum((x-np.asarray(target,dtype=float))**2))
def rastrigin(x):
    x=np.asarray(x,dtype=float); return float(10*len(x)+np.sum(x*x-10*np.cos(2*np.pi*x)))
def softmax_min(vals, temp=1.0):
    vals=np.asarray(vals,dtype=float); w=np.exp(-(vals-vals.min())/temp); return w/w.sum()
print('setup ok')

## Sample around a mean, keep elites, and adapt covariance toward promising directions
Evolution strategies optimize real-valued vectors by sampling perturbations. CMA-ES adds the crucial design choice: learn the covariance of successful steps so future samples align with the valley instead of wandering isotropically.

In [ ]:
# 1) Elite recombination moves the search mean
pts=np.array([[1.,0.],[0.,1.],[2.,2.],[-1.,0.]]); target=np.array([1.,1.])
vals=np.sum((pts-target)**2, axis=1); elite=pts[np.argsort(vals)[:2]]; newmean=elite.mean(axis=0)
plt.figure(figsize=(4,4)); plt.scatter(pts[:,0],pts[:,1],c=vals,cmap='viridis',s=80); plt.scatter(elite[:,0],elite[:,1],facecolors='none',edgecolors='r',s=180,label='elite'); plt.scatter([newmean[0]],[newmean[1]],c='r',marker='x',s=120,label='new mean'); plt.legend(); plt.axis('equal')
plt.title('mean shifts to the best samples')
assert np.allclose(vals, [1.,1.,2.,5.]) and np.allclose(newmean, [0.5,0.5])

In [ ]:
# 2) Covariance learns the direction shared by elites
elite=np.array([[1.,0.],[0.,1.]]); cov=np.cov(elite,rowvar=False,bias=True)+1e-6*np.eye(2)
w, V=np.linalg.eigh(cov); angles=np.linspace(0,2*np.pi,200); ellipse=(V@np.diag(np.sqrt(w))@np.vstack([np.cos(angles),np.sin(angles)])).T+elite.mean(0)
plt.figure(figsize=(4,4)); plt.plot(ellipse[:,0],ellipse[:,1]); plt.scatter(elite[:,0],elite[:,1],c='r'); plt.axis('equal'); plt.title('elite covariance tilts future sampling')
assert cov[0,1] < 0 and abs(cov[0,0]-0.250001)<1e-9

In [ ]:
# 3) On an elongated bowl, an adapted covariance samples along the valley
rng=np.random.default_rng(5); A=np.array([[1.,0.],[0.,25.]]); mean=np.array([2.0,0.0])
round_samples=rng.multivariate_normal(mean,0.25*np.eye(2),size=400)
adapted_samples=rng.multivariate_normal(mean,np.diag([0.7,0.02]),size=400)
round_vals=np.einsum('ij,jk,ik->i',round_samples,A,round_samples); adapted_vals=np.einsum('ij,jk,ik->i',adapted_samples,A,adapted_samples)
plt.figure(figsize=(4,4)); plt.scatter(round_samples[:120,0],round_samples[:120,1],alpha=.25,label='round'); plt.scatter(adapted_samples[:120,0],adapted_samples[:120,1],alpha=.25,label='adapted'); plt.legend(); plt.axis('equal'); plt.title('covariance aligned with the low-curvature direction')
assert np.quantile(adapted_vals,0.1) < np.quantile(round_vals,0.1)

In [ ]:
# 4) Step-size decay trades broad search for final precision
rng=np.random.default_rng(6); means=[]; vals=[]; mean=np.array([4.,-3.])
for gen in range(20):
    sigma=1.2*(0.88**gen); s=mean+rng.normal(0,sigma,(25,2)); val=np.sum(s*s,axis=1); mean=s[np.argsort(val)[:5]].mean(0); means.append(mean.copy()); vals.append(np.sum(mean*mean))
means=np.array(means); plt.figure(figsize=(4,4)); plt.plot(means[:,0],means[:,1],'-o',ms=3); plt.scatter([0],[0],c='r',marker='*',s=120); plt.axis('equal'); plt.title('mean path as sigma shrinks')
assert vals[-1] < vals[0] and np.linalg.norm(means[-1]) < np.linalg.norm(means[0])

In [ ]:
# 5) CMA-style population converges on a 2D target
rng=np.random.default_rng(7); target=np.array([1.,-2.]); mean=np.array([-4.,3.]); cov=2*np.eye(2); start=mean.copy(); best=[]
for _ in range(22):
    samples=rng.multivariate_normal(mean,cov,size=40); val=np.sum((samples-target)**2,axis=1); elite=samples[np.argsort(val)[:8]]; mean=elite.mean(0); cov=np.cov(elite,rowvar=False,bias=True)+0.02*np.eye(2); best.append(val.min())
plt.figure(figsize=(4,4)); plt.scatter([start[0]],[start[1]],label='start'); plt.scatter([mean[0]],[mean[1]],label='final'); plt.scatter([target[0]],[target[1]],c='r',marker='*',s=120,label='target'); plt.legend(); plt.axis('equal'); plt.title('CMA-ES mean reaches the target')
assert np.linalg.norm(mean-target) < np.linalg.norm(start-target) and best[-1] < best[0]